# Level 1: config and data

Creates the seven Apex Precision Manufacturing tables in your catalog and schema. Dates end yesterday so last-week questions stay valid. Runtime is about one minute on serverless.


## Bind catalog, schema, and warehouse

Same widgets as `shared/00_config`. Job parameters override the defaults. The notebook fails if values are still placeholders.


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Generate the seven tables

Writes Parquet to the `raw_data` volume, then Delta tables with comments and informational PK/FK constraints.


In [ ]:
from shared.lib.apex_l1_data import write_l1_tables

counts = write_l1_tables(spark, cfg)
for name, n in counts.items():
    print(f"{name}: {n}")
expected = {
    "plants": 6,
    "production_lines": 24,
    "operators": 400,
    "production_events": 50000,
    "safety_incidents": 40,
    "equipment_feedback": 100,
}
for name, n in expected.items():
    actual = counts[name]
    assert actual == n, f"{name} expected {n} got {actual}"
assert counts["quality_metrics_daily"] == 24 * 365, counts["quality_metrics_daily"]
print("PASS row counts")


## Confirm rolling dates

The latest quality date must be yesterday so shift-start questions work every class.


In [ ]:
from datetime import date, timedelta

row = spark.sql(
    f"SELECT MAX(date) AS max_d, MIN(date) AS min_d FROM {cfg.table('quality_metrics_daily')}"
).collect()[0]
yesterday = date.today() - timedelta(days=1)
assert row["max_d"] == yesterday, f"max date {row['max_d']} is not yesterday {yesterday}"
print("PASS date window", row["min_d"], "to", row["max_d"])
spark.sql(
    f"""CREATE OR REPLACE TABLE {cfg.table('workshop_config')} AS
    SELECT '{cfg.catalog}' AS catalog_name, '{cfg.schema}' AS schema_name, current_timestamp() AS loaded_at"""
)
print("wrote", cfg.table("workshop_config"))
